In [10]:
import numpy as np
import pandas as pd
import sqlite3

In [11]:
#suprime d'abord le contenu existant
with sqlite3.connect("dataimmo.db") as conn:
    cur = conn.cursor()

    tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
    for (table_name,) in tables:
        cur.execute(f"DELETE FROM {table_name}")

In [12]:
geo = pd.read_excel("fr-esr-referentiel-geographique.xlsx", sheet_name="fr-esr-referentiel-geographique",
    dtype={
        "reg_id": str
    })

In [13]:
# selectionne les données REGION (hors doublons)

regs = geo[["reg_id", "reg_nom", "regrgp_nom"]].drop_duplicates()
regs.head()

,reg_id,reg_nom,regrgp_nom
0,R84,Auvergne-Rhône-Alpes,Province
459,R32,Hauts-de-France,Province
1614,R93,Provence-Alpes-Côte d'Azur,Province
2555,R44,Grand Est,Province
3058,R76,Occitanie,Province


In [14]:
# selectionne les données DEPARTEMENT (hors doublons)

deps = geo[["dep_id", "dep_nom", "reg_id"]].drop_duplicates()
deps.head()

,dep_id,dep_nom,reg_id
0,D001,Ain,R84
459,D002,Aisne,R32
1293,D003,Allier,R84
1614,D004,Alpes-de-Haute-Provence,R93
1859,D005,Hautes-Alpes,R93


In [15]:
# selectionne les données ACADEMIE (hors doublons)

acas = geo[["aca_id", "aca_nom", "reg_id"]].drop_duplicates()
acas.head()

,aca_id,aca_nom,reg_id
0,A10,Lyon,R84
459,A20,Amiens,R32
1293,A06,Clermont-Ferrand,R84
1614,A02,Aix-Marseille,R93
2043,A23,Nice,R93


In [16]:
# selectionne les données COMMUNE (hors doublons)

coms = geo[["com_id", "com_nom", "dep_id", "geolocalisation"]].drop_duplicates()
coms.head()

,com_id,com_nom,dep_id,geolocalisation
0,C01001,L'Abergement-Clémenciat,D001,"46.1534255214,4.92611354223"
1,C01002,L'Abergement-de-Varey,D001,"46.0091878776,5.42801696363"
2,C01003,Amareins,D001,NaN
3,C01004,Ambérieu-en-Bugey,D001,"45.9608475114,5.3729257777"
4,C01005,Ambérieux-en-Dombes,D001,"45.9961799872,4.91227250796"


In [17]:
# insere les données dans la base de données
with sqlite3.connect("dataimmo.db") as conn:
    conn.execute("PRAGMA foreign_keys = OFF") # desactive les contraintes le temps de l'insertion
    regs.to_sql("region", conn, if_exists="append", index=False)
    deps.to_sql("departement", conn, if_exists="append", index=False)
    acas.to_sql("academie", conn, if_exists="append", index=False)
    coms.to_sql("commune", conn, if_exists="append", index=False)

In [18]:
communes = pd.read_excel("donnees_communes.xlsx", sheet_name="donnees_communes",
    dtype={
        "CODREG": str,
        "CODDEP": str,
        "CODARR": str,
        "CODCAN": str,
        "CODCOM": str
    })
communes.head()

,CODREG,CODDEP,CODARR,CODCAN,CODCOM,COM,PMUN,PCAP,PTOT
0,84,01,02,08,001,L'Abergement-Clémenciat,779,19,798
1,84,01,01,01,002,L'Abergement-de-Varey,256,1,257
2,84,01,01,01,004,Ambérieu-en-Bugey,14134,380,14514
3,84,01,02,22,005,Ambérieux-en-Dombes,1751,25,1776
4,84,01,01,04,006,Ambléon,112,6,118


In [19]:
# selectionne les données POPULATION

pops = communes[["CODDEP", "CODCOM", "PMUN", "PCAP"]].copy()
#pops["com_id"] = "C" + pops["CODDEP"] + pops["CODCOM"] # pas un simple plus !!
# code INSEE
pops["com_id"] = np.where(
    pops["CODDEP"].str.len() == 2,
    "C" + pops["CODDEP"] + pops["CODCOM"],
    "C" + pops["CODDEP"] + pops["CODCOM"].str[-2:]
)# metropole / DOM
pops.pop("CODDEP")
pops.pop("CODCOM")
pops.head()

,PMUN,PCAP,com_id
0,779,19,C01001
1,256,1,C01002
2,14134,380,C01004
3,1751,25,C01005
4,112,6,C01006


In [20]:
with sqlite3.connect("dataimmo.db") as conn:
    conn.execute("PRAGMA foreign_keys = OFF") # desactive les contraintes le temps de l'insertion
    conn.execute("delete from population")
    pops.to_sql("population", conn, if_exists="append", index=False)

In [21]:
colonnes_obligatoires = [
    "date_mutation", "valeur_fonciere"
]

val = pd.read_excel("Valeurs-foncières.xlsx", sheet_name="Feuil1",
    dtype={
        "Code commune": str,
        "Code departement": str,
        "Code voie": str
    }
    ).rename(columns={
    "No voie": "no_voie",
    "Type de voie": "type_voie",
    "Voie": "voie",
    "Code voie": "code_voie",
    "B/T/Q": "btq",
    "Code postal": "code_postal",
    "Code commune": "code_com",
    "Code departement": "code_dep",
    "Surface Carrez du 1er lot": "surface_carrez",
    "Type local": "type_local",
    "Surface reelle bati": "surface_reelle_bati",
    "Nombre pieces principales": "nombre_pieces",
    "Surface terrain": "surface_terrain",
    "Nature culture": "nature_culture",
    "Date mutation": "date_mutation",
    "Valeur fonciere": "valeur_fonciere",
    "Section": "section_cadastrale",
    "No plan": "no_plan"
}).dropna(subset=colonnes_obligatoires) # supprime les lignes avec des valeurs manquantes pour simplifier l'insertion

In [23]:
with sqlite3.connect("dataimmo.db") as conn:
    conn.execute("PRAGMA foreign_keys = OFF") # desactive les contraintes le temps de l'insertion
    conn.execute("delete from vente")
    conn.execute("delete from adresse")
    conn.execute("delete from bien")

    def safe_val(val):
        """Convertit les valeurs pandas/numpy pour SQLite"""
        if pd.isna(val):
            return None
        if isinstance(val, (pd.Timestamp, pd.Timedelta)):
            return str(val)  # convertir en string
        if isinstance(val, (np.integer, np.int64, np.int32)):
            return int(val)
        if isinstance(val, (np.floating, np.float64, np.float32)):
            return float(val)
        return val

    # itere sur les lignes du dataframe
    # obtient les données de ADRESSE, VENTE et BIEN
    adresse_id = 0
    bien_id = 0
    i = 0

    for index, line in val.iterrows(): 
        adr = line[["no_voie", "type_voie", "voie", "code_postal", "code_com", "code_dep", "code_voie", "btq"]]
        bien = line[["type_local", "surface_reelle_bati", "nombre_pieces", "surface_terrain", "surface_carrez", "nature_culture"]]
        vente = line[["date_mutation", "valeur_fonciere", "section_cadastrale", "no_plan"]]

        # pour faire le lien avec la table commune, on utilise le code commune et le code departement
        com_id = "C" + adr["code_dep"] + adr["code_com"].zfill(3) if len(adr["code_dep"]) == 2 else "C" + adr["code_dep"] + (adr["code_com"])[-2:] # metropole / DOM

        # insert l'adresse si elle n'existe pas deja (AK)
        result = conn.execute("SELECT adresse_id FROM adresse WHERE no_voie = ? AND code_voie = ? AND btq = ? AND com_id = ? LIMIT 1", (safe_val(adr["no_voie"]), safe_val(adr["code_voie"]), safe_val(adr["btq"]), safe_val(com_id))).fetchone()
        if result is not None:
            adresse_id = result[0]
        else:
            cur = conn.cursor()
            cur.execute(
                """
                INSERT INTO adresse (no_voie, code_voie, btq, type_voie, voie, code_postal, com_id)
                VALUES (?, ?, ?, ?, ?, ?, ?)
                """,
                (safe_val(adr["no_voie"]), safe_val(adr["code_voie"]), safe_val(adr["btq"]), safe_val(adr["type_voie"]), safe_val(adr["voie"]), safe_val(adr["code_postal"]), safe_val(com_id))
            )
            conn.commit()
            adresse_id = cur.lastrowid

        # insert le bien si il n'existe pas deja
        result = conn.execute("SELECT bien_id FROM bien WHERE type_local = ? AND surface_reelle_bati = ? AND nombre_pieces = ? AND surface_terrain = ? AND surface_carrez = ? AND nature_culture = ? LIMIT 1",  (safe_val(bien["type_local"]), safe_val(bien["surface_reelle_bati"]), safe_val(bien["nombre_pieces"]), safe_val(bien["surface_terrain"]), safe_val(bien["surface_carrez"]), safe_val(bien["nature_culture"]))).fetchone()
        if result is not None:
            bien_id = result[0]
        else:
            cur = conn.cursor()
            cur.execute(
                """
                INSERT INTO bien (type_local, surface_reelle_bati, nombre_pieces, surface_terrain, surface_carrez, nature_culture)
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (safe_val(bien["type_local"]), safe_val(bien["surface_reelle_bati"]), safe_val(bien["nombre_pieces"]), safe_val(bien["surface_terrain"]), safe_val(bien["surface_carrez"]), safe_val(bien["nature_culture"]))
            )
            conn.commit()
            bien_id = cur.lastrowid

        vente["adresse_id"] = adresse_id
        vente["bien_id"] = bien_id

        cur = conn.cursor()
        cur.execute(
            """
            INSERT INTO vente (date_mutation, valeur_fonciere, section_cadastrale, no_plan, adresse_id, bien_id)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (safe_val(vente["date_mutation"]), safe_val(vente["valeur_fonciere"]), safe_val(vente["section_cadastrale"]), safe_val(vente["no_plan"]), safe_val(adresse_id), safe_val(bien_id))
        )
        conn.commit()
        
        #if i > 10: # limiter à 1000 lignes pour ne pas surcharger la base de données
        #    break
        i += 1